# MD Validation and Performance Diagnostics

For MD, speed is meaningless until forces and stability are trustworthy.  The
basic checks are:

$$
F_i^\alpha \approx -\frac{E(r_i^\alpha+\epsilon)-E(r_i^\alpha-\epsilon)}{2\epsilon},
\qquad
\Delta E_\mathrm{NVE}(t)=E(t)-E(0).
$$

This workbook turns those checks into plots instead of buried test output.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import mlx.core as mx
import numpy as np


def find_repo_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("could not locate repository root")


ROOT = find_repo_root()


## What a force check is proving

For any differentiable potential energy \(E(r)\), the Cartesian force component
is

$$
F_i^\alpha = -\frac{\partial E}{\partial r_i^\alpha}.
$$

The central-difference estimate used here is

$$
F_{i,\mathrm{fd}}^\alpha
=-\frac{
E(r_i^\alpha+\epsilon)-E(r_i^\alpha-\epsilon)
}{2\epsilon}
+\mathcal O(\epsilon^2).
$$

The validation report records two complementary errors:

$$
e_\mathrm{max}=\max_{i,\alpha}
\left|F_i^\alpha-F_{i,\mathrm{fd}}^\alpha\right|,
\qquad
e_\mathrm{rms}=
\sqrt{\frac{1}{3N}
\sum_{i,\alpha}
\left(F_i^\alpha-F_{i,\mathrm{fd}}^\alpha\right)^2 }.
$$

`max` is good for finding a single bad coordinate.  `rms` is better for seeing
whether the whole force field is noisy.


## Finite-difference force checks

Each force term is compared against central finite differences on a seeded toy
geometry.  The bar plot makes the outlier coordinate visible.


In [ ]:
from mlx_atomistic.validation import run_force_validation_suite, summarize_validation_results

validation = run_force_validation_suite(cases_per_term=1, epsilon=1e-3, tolerance=5e-3)
summary = summarize_validation_results(validation)
rows = [item.to_dict() for item in validation]
print(summary)

labels = [row["case_name"] for row in rows]
errors = [row["max_abs_error"] for row in rows]
tolerances = [row["tolerance"] for row in rows]

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(labels, errors, label="max |F - F_fd|")
ax.plot(labels, tolerances, color="tab:red", marker="o", label="tolerance")
ax.set_yscale("log")
ax.set_ylabel("force error")
ax.set_title("finite-difference force validation")
ax.tick_params(axis="x", rotation=30)
ax.legend()
fig.tight_layout()


## LJ liquid smoke trajectory

The neighbor list should preserve the same qualitative physics while reducing
the number of evaluated pairs.  The following small run is intentionally quick:
it checks finite energies, pair counts, rebuild counts, temperature, and drift.


### Neighbor-list model

The all-pairs nonbonded loop has \(\mathcal O(N^2)\) candidate pairs:

$$
N_\mathrm{pairs} = \frac{N(N-1)}{2}.
$$

With a cutoff \(r_c\), only nearby pairs are physically evaluated.  A Verlet
neighbor list uses a larger search radius \(r_c+r_\mathrm{skin}\), then reuses
that list until atoms have moved far enough:

$$
\max_i |\Delta r_i| > \frac{1}{2}r_\mathrm{skin}.
$$

The pair-count plot is a direct proxy for nonbonded work.  The rebuild-count
plot tells us how often we pay the more expensive list-construction cost.


In [ ]:
from mlx_atomistic import Cell
from mlx_atomistic.md import LennardJonesPotential, SimulationConfig, simulate_nve
from mlx_atomistic.neighbors import NeighborListManager

def cubic_lattice(n_side: int, spacing: float) -> np.ndarray:
    grid = np.array(np.meshgrid(*(np.arange(n_side) for _ in range(3)), indexing="ij"))
    return (grid.reshape(3, -1).T + 0.5).astype(np.float32) * spacing

rng = np.random.default_rng(11)
positions = cubic_lattice(3, 1.55)
velocities = rng.normal(scale=0.04, size=positions.shape).astype(np.float32)
velocities -= velocities.mean(axis=0, keepdims=True)
masses = np.ones(positions.shape[0], dtype=np.float32)
cell = Cell.orthorhombic((5.5, 5.5, 5.5))
potential = LennardJonesPotential(cutoff=2.5, shift=True)
neighbors = NeighborListManager(cell=cell, cutoff=2.5, skin=0.4)

lj = simulate_nve(
    positions,
    velocities,
    masses=masses,
    cell=cell,
    force_terms=potential,
    neighbor_manager=neighbors,
    config=SimulationConfig(dt=0.002, steps=250, sample_interval=10),
)
mx.eval(lj.total_energy, lj.temperature, lj.pair_count, lj.rebuild_count)

time = np.arange(lj.total_energy.shape[0]) * 0.002
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].plot(time, np.asarray(lj.energy_drift))
axes[0].set_title("NVE energy drift")
axes[0].set_xlabel("time")
axes[0].set_ylabel("E(t) - E(0)")
axes[1].plot(time, np.asarray(lj.temperature))
axes[1].set_title("instantaneous temperature")
axes[1].set_xlabel("time")
axes[2].plot(time, np.asarray(lj.pair_count), label="pairs")
axes[2].plot(time, np.asarray(lj.rebuild_count), label="rebuilds")
axes[2].set_title("neighbor diagnostics")
axes[2].set_xlabel("time")
axes[2].legend()
fig.tight_layout()


## What this tells us

For this library, MD quality should be judged on three axes: force correctness,
energy/temperature stability, and pair-path scaling.  The tests enforce these
numerically; this notebook shows the same evidence visually.


### Practical interpretation

Use this notebook when changing force terms, neighbor-list logic, or integrator
details:

- if force errors jump, inspect the failing atom/axis before benchmarking;
- if NVE drift grows monotonically, reduce `dt` or inspect force consistency;
- if pair counts stay close to all-pairs counts, the cutoff/cell setup is not
  giving the neighbor list much opportunity to help;
- if rebuilds happen every step, the `skin` is too small for the timestep and
  temperature.
